# NxM validation: held-out detector pulses

This notebook applies the regularized 12-channel fit to a raw series not retained in the saved PCA training sample. The ignored `data` symlink is required to reproduce the recorded results.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import hashlib
import os
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np

from importlib import reload


REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))
import nxm
from nxm import NxMReconstructor
import pycrp
reload(pycrp)
reload(nxm)


paths = {
    'templates': REPO_ROOT / 'data/pca/templates_pca_543.joblib',
    'csd': REPO_ROOT / 'data/final_psd_and_templates/csd_results.joblib',
    'training': REPO_ROOT / 'data/pca/traces_pca_543.joblib',
    'candidate': REPO_ROOT / 'data/trace_signal/trace_z1_23231216_233807.joblib',
}
DATA_AVAILABLE = all(path.exists() for path in paths.values())
if not DATA_AVAILABLE:
    missing = [str(path) for path in paths.values() if not path.exists()]
    print('Held-out validation skipped. Restore the ignored data symlink and these artifacts:')
    print('\n'.join(missing))

## Load artifacts and prove the holdout split

In [ ]:
if DATA_AVAILABLE:
    templates = joblib.load(paths['templates'], mmap_mode='r')
    csd_artifact = joblib.load(paths['csd'], mmap_mode='r')
    training = joblib.load(paths['training'], mmap_mode='r')
    candidate = joblib.load(paths['candidate'], mmap_mode='r')
    channels = list(csd_artifact['channels_ordered'])

    def fingerprint(trace):
        return hashlib.blake2b(np.asarray(trace).tobytes(), digest_size=16).digest()

    training_ids = {fingerprint(trace) for trace in training[channels[0]]}
    overlap = [index for index, trace in enumerate(candidate[channels[0]]) if fingerprint(trace) in training_ids]
    held_out_indices = [index for index in range(len(candidate[channels[0]])) if index not in overlap]
    print(f'Candidate events: {len(candidate[channels[0]])}')
    print(f'PCA-training overlap: {overlap}')
    print(f'Held-out events: {len(held_out_indices)}')
    assert overlap == [9]
    assert len(held_out_indices) == 26

## Fit all held-out events

The fit uses 4,000 low-frequency RFFT bins, the full 12-channel CSD, a ±150-sample initialization search, and a ±10-sample final window. Independent events are evaluated concurrently; the fit itself remains deterministic.

In [ ]:
if DATA_AVAILABLE:
    means = np.stack([templates['template_avg'][channel] for channel in channels])
    components = np.stack([templates['template_pca'][channel] for channel in channels])
    variances = np.stack([templates['variance_pca'][channel] for channel in channels])
    csd = np.asarray(csd_artifact['csd_matrices'])
    
    # Build NxMReconstructor for all channels, and for each channel individually (1xM)
    reconstructor = nxm.NxMReconstructor(
        means, components, variances, csd,
        channel_names=channels, frequency_limit=4000,
    )
    mean_only = nxm.NxMReconstructor(
        means, np.empty((len(channels), 0, means.shape[1])),
        np.empty((len(channels), 0)), csd,
        channel_names=channels, frequency_limit=4000,
    )
    reconstructor_singles = [nxm.NxMReconstructor(
        means[ich:ich+1], components[ich:ich+1], variances[ich:ich+1], csd[:, ich:ich+1, ich:ich+1],
        ) for ich in range(len(channels))]
    
    # Build OF filters for each channel
    t0_mean = templates["metadata"]["template_t0"]
    OFs = [pycrp.filter.OF(means[i], csd[:,i,i].real, fs = 1, lowfreq_chi2_bins =100) for i in range(len(channels))]
    
    # Get data
    traces = []
    for event_index in held_out_indices:
        event = np.stack([np.asarray(candidate[channel][event_index], dtype=float) for channel in channels])
        event -= np.mean(event[:, :16000], axis=1, keepdims=True)
        traces.append(event)


In [ ]:
ievent = 12

event = traces[ievent]
result_total, results, model_total, models = pycrp.fit.fit_channels(event, csd, len_fit=1050, t0_guess=16030, tau_rise_guess=20, tau_fall_guess=50, t0_bound=50, x_scale=[20,1,2], nfall=1)

In [ ]:
for ich, ch in enumerate(channels):
    print(f'Channel {ch}: t0 = {results[ich]["params"]["t0"]}, t0_mean = {t0_mean}, shift = {int(results[ich]["params"]["t0"]-t0_mean)}')

In [ ]:

init_delays = []
init_amplitudes = []
for ich, ch in enumerate(channels):
    delay = int(results[ich]["params"]["t0"]-t0_mean)
    trace = np.roll(event[ich], -delay)
    init_delays.append(delay)
    init_amplitudes.append(OFs[ich].of0(trace)[0])


amplitudes, delays = reconstructor.initialize(event, (-150, 150))
result_fit_nxm =  reconstructor.fit(
    event, initial_amplitudes=amplitudes, initial_delays=delays,
    delay_half_width=10,
    optimizer_options={'xtol': 0.1, 'ftol': 1e-2, 'maxiter': 20, 'maxfev': 1400},
)

In [ ]:
init_delays = []
init_amplitudes = []
event_traces = []
for ich, ch in enumerate(channels):
    delay = int(results[ich]["params"]["t0"]-t0_mean)
    trace = np.roll(event[ich], -delay)
    init_delays.append(delay)
    init_amplitudes.append(OFs[ich].of0(trace)[0])


# amplitudes, delays = reconstructor.initialize(event, (-150, 150))
# result_fit_nxm =  reconstructor.fit(
#     event, initial_amplitudes=amplitudes, initial_delays=delays,
#     delay_half_width=10,
#     optimizer_options={'xtol': 0.1, 'ftol': 1e-2, 'maxiter': 20, 'maxfev': 1400},
# )


for ich, ch in enumerate(channels):
    delay = int(results[ich]["params"]["t0"]-t0_mean)
    trace = np.roll(event[ich], -delay)
    amplitude = init_amplitudes[ich]
    
    # Run 1xm once with initial parameters
    profile, model_fft = reconstructor_singles[ich]._profile_fft(np.array([np.fft.rfft(trace)]).T, np.array([amplitude]), np.array([0]))
    model_1xm = np.fft.irfft(model_fft.T[0])
    
    # Run 1xm fit
    fit_result = reconstructor_singles[ich].fit(
            np.array(trace)[None,:], initial_amplitudes= np.array([amplitude]), initial_delays=np.array([0]),
            delay_half_width=10,
            optimizer_options={'xtol': 0.1, 'ftol': 1e-2, 'maxiter': 20, 'maxfev': 1400},
        )
    
    model_1xm_fit = reconstructor_singles[ich].reconstruct(fit_result)
    
    # Print difference of t0 and amplitude between initial and fitted 1xM model
    print(f'Init t0 = 0, amp = {amplitude}, chi2 = {profile.objective/16384}')
    print(f'Fit t0 = {fit_result.delays[0]}, amp = {fit_result.amplitudes[0]}, chi2 = {fit_result.objective/16384}')

    plt.plot(trace, label=f'Data')
    plt.plot(np.roll(models[ich], -delay), label=f'Fit (analytical)')
    plt.plot(means[ich]*amplitude, label=f'Fit (OF0, amp: {amplitude:.2f})')
    plt.plot(model_1xm, label=f'1xM init')
    plt.plot(model_1xm_fit[0], label=f'Fit (1xM)')
    plt.xlim(15900, 16800)
    plt.xlabel('Sample Index')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.show()
    # break
    

In [ ]:
fit_result